In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import random
import sys

In [7]:
df = pd.read_csv('/content/train (4).csv', on_bad_lines='skip', engine='python')
text = " ".join(df['text'].dropna().astype(str)).lower()

print(f'Total characters in text: {len(text)}')

Total characters in text: 15990357


In [8]:
vocab = sorted(set(text))
print(f'Vocabulary size: {len(vocab)}')

char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])

Vocabulary size: 92


In [9]:
seq_length = 100

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

dataset = sequences.map(split_input_target)

BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

In [10]:
vocab_size = len(vocab)
embedding_dim = 64
rnn_units = 128

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_shape=(None,)),
    tf.keras.layers.LSTM(rnn_units, return_sequences=True, recurrent_initializer='glorot_uniform'),
    tf.keras.layers.Dense(vocab_size)
])


def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 64)       │         5,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 128)      │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 92)       │        11,868 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,572 (455.36 KB)

 Trainable params: 116,572 (455.36 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
EPOCHS = 20
history = model.fit(dataset, epochs=EPOCHS)

Epoch 1/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 49s 18ms/step - loss: 2.0132
Epoch 2/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 43s 17ms/step - loss: 1.6103
Epoch 3/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 42s 16ms/step - loss: 1.4909
Epoch 4/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 43s 17ms/step - loss: 1.4355
Epoch 5/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - loss: 1.4029
Epoch 6/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 42s 16ms/step - loss: 1.3808
Epoch 7/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 82s 17ms/step - loss: 1.3644
Epoch 8/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 42s 16ms/step - loss: 1.3519
Epoch 9/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 43s 17ms/step - loss: 1.3418
Epoch 10/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 42s 16ms/step - loss: 1.3336
Epoch 11/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 41s 16ms/step - loss: 1.3266
Epoch 12/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - loss: 1.3207
Epoch 13/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - loss: 1.3156
Epoch 14/20
2473/2473 ━━━━━━━━━━━━━━━━━━━━ 42s 16ms/step - loss: 1.3111
E

In [12]:
def generate_text(model, start_string, num_generate=100, temperature=1.0):
    input_eval = [char2idx.get(s, 0) for s in start_string.lower()]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    model.layers[1].reset_states()

    for _ in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0) / temperature
        predicted_id = tf.random.categorical(
            predictions, num_samples=1)[-1, 0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)


print(generate_text(model, start_string="The ", num_generate=200, temperature=0.8))

The dg ing o  p’s havexp. go inecate at as touranox is alaron canched cansedern tomanan tid ed pritorwive teeud s gepal tidein, tsthe arthes cinderut shititerecthed is he ane mive fooongr wede in peang k 
